# Week 10 extension — train + evaluate (notebook 10d)

This notebook merges the **training** half (Week 10's Tasks 60–66, formerly
in `10d_extend_and_train.ipynb`) and the **evaluation** half (Tasks 67–70,
formerly in `10e_diffusion_NLL_ablations.ipynb`) into a single Colab-friendly
file. Each Colab session is isolated, so doing both halves in one notebook
means a fresh runtime can pick up where you left off without re-running
setup twice and without risking the `EXPERIMENTS` dict drifting between the
two halves.

## How to use this notebook

There is **one flag at the top** that determines what the notebook does:

- `MODE = "train"` — runs the data-augmentation cells (Tasks 60–63), the
  experiment-menu setup (Tasks 64–65), the training loop (Task 66), and the
  visual sanity check. The training loop is **idempotent**: it skips any
  experiment whose `ckpt_<name>.ckpt` already exists. Run the notebook many
  times with one new entry enabled in `ENABLED_EXPERIMENTS` per session.
- `MODE = "eval"` — skips training, jumps into the NLL ablation pipeline
  (Tasks 67–69), and scores every `ckpt_E*.ckpt` it finds in this directory.

**Recipe:** run with `MODE = "train"` repeatedly (one new experiment per
session) until you have all the checkpoints you want, then flip to
`MODE = "eval"` and run the notebook once to score them.

[**↓ Jump to evaluation (Tasks 67–70)**](#eval-mode)

> The **test split is reserved for the PI**. Every data-loading cell in this
> notebook filters to `split in {"train", "val"}`. Do not change that.


In [ ]:
MODE = "train"   # set to "eval" once you have ckpt_E*.ckpt files to score

assert MODE in ("train", "eval"), f"MODE must be 'train' or 'eval', got {MODE!r}"
TRAIN_MODE = (MODE == "train")
EVAL_MODE  = (MODE == "eval")
print(f"MODE = {MODE!r}  (TRAIN_MODE={TRAIN_MODE}, EVAL_MODE={EVAL_MODE})")


In [ ]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb


In [ ]:
# Bootstrap sys.path and locate artifacts.
import os, sys

for _p in [os.path.abspath(os.path.join(repo_path, "weeks", "week_10")),
           os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from conditioned_infrastructure import find_week10_artifacts
# parquet_v2 is built fresh in Part A every run; raw CSV is needed by the
# eval phase (Task 67 Part 1). Both are listed as required so a missing
# raw CSV fails loudly at setup time rather than at Task 67.
paths = find_week10_artifacts(extra_required=[
    "data/composite_sunspot_groups_peak_area.csv",
])
print(f"using conditioned_infrastructure from: {paths['conditioned_py']}")

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    ExtendedConditionalDiffusionLightning,
    train_experiment,
    load_trained_experiment,
    sample_conditional_extended,
    build_model,
    block_cond_concat,
    k_run_combined,
    discover_experiment_checkpoints,
)
from butterflAI_model import ButterflAIModel

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the v1 parquet — we extend it but never modify it.
# Test split is reserved for the PI.
windows_v1 = pd.read_parquet(paths["parquet_v1"])
windows_v1 = windows_v1.loc[windows_v1["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v1 parquet (train+val only): {len(windows_v1)} rows")
print(f"  splits: {windows_v1['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v1['cycle'].unique())}")
print(f"v2 parquet target: {PARQUET_V2}")
print(f"device           : {device}")


---
## Part A — Build the augmented parquet

We're going to add three families of new conditioning columns to the
v1 parquet:

1. **Cycle and hemisphere identifiers** — a normalized cycle number and
   a north/south indicator.
2. **Opposite-hemisphere summaries** — for each window, the
   *contemporaneous* opposite-hemisphere activity. This is not leakage:
   contemporaneous opposite-hemisphere activity is operationally
   observable (an operational forecaster on the day of the same window
   would have it).
3. **Smoothed-area trajectory** — the past `K` windows of `area_smoothed`
   for the same hemicycle, as a short autoregressive history.

The build cells **rewrite `diffusion_windows_v2.parquet` every run**. We
deliberately do *not* gate on file existence — if you change how a
column is computed and don't see the change downstream, the most
common explanation is "the file was cached." We avoid that failure
mode by always rewriting.


---
## Task 60 — Cycle and hemisphere identifiers

Add two new columns to the dataframe:

- `cycle_norm`: cycle number rescaled to roughly `[-1, +1]` over the
  train range. The point of normalization is not to be exactly in
  `[-1, +1]` — it's to put the input on the same numerical scale as the
  other conditioning vectors so the network doesn't have to learn an
  outsized weight for it.
- `hemi_id`: `+1` for north, `-1` for south.

Both are cheap and let downstream experiments test whether
*structural* per-cycle / per-hemisphere effects survive once amplitude
is controlled for.


In [ ]:
# Task 60 — add cycle_norm and hemi_id.

windows_aug = windows_v1.copy()

# TODO: compute cycle_norm. Use train-set cycle range so this is well
# defined for both splits. Aim for the train cycle range to map roughly
# onto [-1, +1].
_train_cycles = windows_aug.loc[windows_aug["split"] == "train", "cycle"]
_cmin, _cmax  = _train_cycles.min(), _train_cycles.max()
windows_aug["cycle_norm"] = 2.0 * (windows_aug["cycle"] - _cmin) / (_cmax - _cmin) - 1.0

# TODO: compute hemi_id. +1 north, -1 south.
windows_aug["hemi_id"] = np.where(windows_aug["hemisphere"] == "north", 1.0, -1.0)

print("cycle_norm range:", windows_aug["cycle_norm"].min(), windows_aug["cycle_norm"].max())
print("hemi_id values  :", windows_aug["hemi_id"].unique())


---
## Task 61 — Opposite-hemisphere conditioning

For each window in hemisphere *h* at time `tau_center`, attach the
contemporaneous opposite-hemisphere activity summary:

- `opp_area_smoothed` — opposite hemisphere's `area_smoothed`
- `opp_mu_universal`  — opposite hemisphere's `mu_universal`
- `opp_amplitude`     — opposite hemisphere's `amplitude`
- `opp_valid`         — 1 if a matching opposite row was found at the
  same `(cycle, tau_center)`, else 0.

When `opp_valid == 0` (no matching opposite row), impute with the
**train-set mean** of each opposite-* column. This way the network always
sees a defined input; downstream you can decide whether to gate on the
mask.

**Implementation hint:** the cleanest way is a self-merge of the
dataframe with itself: produce a "right side" with `hemisphere`
flipped and renamed columns, then merge on `(cycle, tau_center)`.


In [ ]:
# Task 61 — attach contemporaneous opposite-hemisphere summaries.

# TODO: build a "right side" dataframe with hemisphere flipped and the
#       three columns we want renamed with the "opp_" prefix.
_flip = {"north": "south", "south": "north"}
_right = (windows_aug
          .loc[:, ["cycle", "tau_center", "hemisphere",
                   "area_smoothed", "mu_universal", "amplitude"]]
          .assign(opp_of_hemisphere=lambda d: d["hemisphere"].map(_flip))
          .drop(columns=["hemisphere"])
          .rename(columns={
              "area_smoothed": "opp_area_smoothed",
              "mu_universal":  "opp_mu_universal",
              "amplitude":     "opp_amplitude",
              "opp_of_hemisphere": "hemisphere",
          }))

# TODO: merge on (cycle, tau_center, hemisphere) so each row gets the
#       opposite-side row that has the *flipped* hemisphere stored under
#       the same hemisphere key.
windows_aug = windows_aug.merge(
    _right, on=["cycle", "tau_center", "hemisphere"],
    how="left", indicator="_opp_match",
)
windows_aug["opp_valid"] = (windows_aug["_opp_match"] == "both").astype(np.float32)
windows_aug = windows_aug.drop(columns=["_opp_match"])

# TODO: impute missing opp_* values with train-set means.
_opp_cols = ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"]
_train_mask = (windows_aug["split"] == "train") & (windows_aug["opp_valid"] == 1.0)
for _c in _opp_cols:
    _mean = windows_aug.loc[_train_mask, _c].mean()
    windows_aug[_c] = windows_aug[_c].fillna(_mean)

print(f"opp_valid coverage: {windows_aug['opp_valid'].mean():.3f}")
print(f"opp_area_smoothed (train, valid): "
      f"mean={windows_aug.loc[_train_mask, 'opp_area_smoothed'].mean():.3f}, "
      f"std={windows_aug.loc[_train_mask, 'opp_area_smoothed'].std():.3f}")


---
## Task 62 — Smoothed-area trajectory (cycle history)

For each window, attach the previous `K` values of `area_smoothed`
from the same hemicycle (same `cycle` AND same `hemisphere`), ordered
chronologically by `tau_center`. Columns: `area_lag1`, `area_lag2`, …,
`area_lagK`.

`area_lag1` is the *previous* window's smoothed area; `area_lagK` is
the one K steps back. Boundary windows (near the start of a hemicycle,
where fewer than K prior windows exist) get train-set-mean imputation
and a `traj_valid` column set to 0 for that row.

The point: cycle *history* is information that amplitude alone misses.
A window 6 months into a strong cycle and a window 6 months from the
end of a strong cycle have similar amplitude but very different
trajectory.

**Implementation hint:** sort within each hemicycle by `tau_center`,
then shift the `area_smoothed` series by 1, 2, …, K.


In [ ]:
# Task 62 — attach the smoothed-area trajectory (K lagged values).

K_LAGS = 4

# TODO: within each (cycle, hemisphere) group, sort by tau_center and
#       produce K columns of shifted area_smoothed.
_sorted = windows_aug.sort_values(["cycle", "hemisphere", "tau_center"])

_lag_cols = [f"area_lag{k}" for k in range(1, K_LAGS + 1)]
for k, col in enumerate(_lag_cols, start=1):
    _sorted[col] = _sorted.groupby(["cycle", "hemisphere"])["area_smoothed"].shift(k)

# A row is traj_valid iff *all* K lags exist.
_sorted["traj_valid"] = (~_sorted[_lag_cols].isna().any(axis=1)).astype(np.float32)

# TODO: impute boundary NaNs with train-set means.
_train_mask = (_sorted["split"] == "train") & (_sorted["traj_valid"] == 1.0)
for col in _lag_cols:
    _mean = _sorted.loc[_train_mask, col].mean()
    _sorted[col] = _sorted[col].fillna(_mean)

windows_aug = _sorted.sort_index()  # restore original row order

print(f"traj_valid coverage: {windows_aug['traj_valid'].mean():.3f}")
print(f"lag columns: {_lag_cols}")


---
## Task 63 — Write `diffusion_windows_v2.parquet` and sanity-check

Write the augmented dataframe. Sanity checks before we trust it
downstream:

- Row count unchanged from v1 (we did not gain or lose any windows).
- Every original v1 column is preserved bit-for-bit.
- New cond columns are finite **wherever the validity mask says they
  should be**.
- `split` column is unchanged.


In [ ]:
# Task 63 — write v2 + sanity checks.

# Sanity 1: row count.
assert len(windows_aug) == len(windows_v1), \
    f"row count drifted: {len(windows_aug)} vs v1 {len(windows_v1)}"

# Sanity 2: every v1 column preserved bit-for-bit.
for c in windows_v1.columns:
    assert c in windows_aug.columns, f"missing v1 column {c}"
    if windows_v1[c].dtype.kind in "fc":
        assert np.allclose(windows_v1[c].to_numpy(),
                           windows_aug[c].to_numpy(), equal_nan=True), c
    else:
        assert (windows_v1[c].astype(str).to_numpy()
                == windows_aug[c].astype(str).to_numpy()).all(), c

# Sanity 3: new columns finite where the validity masks allow.
NEW_COND_COLS = ["cycle_norm", "hemi_id",
                 "opp_area_smoothed", "opp_mu_universal", "opp_amplitude",
                 *[f"area_lag{k}" for k in range(1, K_LAGS + 1)]]
for c in NEW_COND_COLS:
    assert windows_aug[c].notna().all(), f"NaN remains in {c}"

# Sanity 4: split column unchanged.
assert (windows_aug["split"].to_numpy() == windows_v1["split"].to_numpy()).all()

# Always rewrite. Never cache.
windows_aug.to_parquet(PARQUET_V2, index=False)
print(f"wrote {PARQUET_V2}  ({len(windows_aug)} rows, {len(windows_aug.columns)} cols)")
print(f"new cond columns: {NEW_COND_COLS}")


---
## Part B — wandb setup and the experiment menu

### Task 64 — Per-student wandb project

Each student gets their **own** wandb project. Replace the placeholder
strings below with your handle. Every training run in this notebook
logs to that project with the experiment ID as the run name; you can
compare all your variants on a single dashboard.

If wandb is unavailable in your environment, training will fall back to
a local CSV logger automatically. The assertion guard runs in both
modes so eval mode also tells you if you forgot to personalize the
project name.


In [ ]:
# Task 64 — wandb identity. EDIT THESE.

WANDB_PROJECT = "butterflai-w10ext-<your-handle>"
WANDB_ENTITY  = "<your-wandb-username>"      # set to None if you don't use teams

assert "your-handle" not in WANDB_PROJECT, \
    "Set WANDB_PROJECT to your own project name before training."


### Task 65 — Design your own experiments

The Week 10 baseline (E0) reproduces the existing conditional
diffusion on the v2 parquet — no new knobs. Everything beyond it is
your call. Each variant you propose should change **one knob** from
the previous run and answer **one question**.

The knobs available are:

- **Cond groups** (`groups` / `consumed_keys`): `base`, plus any of
  `cyclehemi`, `opp`, `traj`.
- **Architecture** (`arch`): `concat` or `film`.
- **Classifier-free guidance**: `cond_dropout_p=0.1` at training time;
  10e sweeps the guidance weight at sampling.
- **Fourier lifting**: `fourier=True` lifts cond scalars via sin/cos.

The menu below escalates roughly by effort-per-insight. Pick what's
interesting, add a new entry to `EXPERIMENTS`, and progress one
variant per session.

**Level 1 — same cond, change the channel.**
Add one of the new cond groups (`cyclehemi`, `opp`, `traj`) to E0's
`consumed_keys`. *Does the diffusion's val NLL drop when given more
information, with the architecture held fixed?*

**Level 2 — same information, change the mechanism.**
Switch `arch` from `concat` to `film` while keeping `cond_base` only.
*Does the modulation mechanism alone close the gap with classical?*

**Level 3 — best information × best mechanism.**
Combine your best Level 1 cond set with FiLM. *Is the combined gain
additive, or did Level 2 already capture it?*

**Level 4 — guidance.**
Set `cond_dropout_p=0.1` and train. Sampling guidance is swept in 10e.
*Can sharpening the conditional density buy you margin over Level 3?*

**Level 5 — Fourier lifting.**
Set `fourier=True`. *Does sin/cos lifting of the cond scalars help the
network represent boundaries?*

You can go further — bump `hidden_dim` / `n_layers`, raise `K_LAGS`
back in Task 62, pair lagged opposite-hemisphere with trajectory, or
anything else you can defend. Different students should diverge here;
results pool in 10e.

In [ ]:
# Task 65 — experiment specs. Start with the baseline; add new entries
# below as you escalate (see the markdown above). See
# conditioned_infrastructure.build_model for the recognized keys.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     20000,
    "lr":             1e-3,
    "batch_size":     64,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    "E0": _spec(),   # baseline — same 4-D cond on the v2 parquet
    # Add your own variants below, e.g.:
    #   "E1": _spec(consumed_keys=["cond_base", "cond_opp"],
    #               groups=["base", "opp"]),
    #   "E2": _spec(arch="film"),
}

for name, cfg in EXPERIMENTS.items():
    print(f"{name}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")

---
## Part C — Disciplined sweep (TRAIN MODE)

### Task 66 — Enable a subset and train

Discipline:

- Add at most **one new experiment per session** beyond the baseline.
  Two-knob-at-a-time changes make the 10e diff impossible to read.
- Each enabled experiment logs to your wandb project under its name
  (`E0`, `E1`, …); compare them on a single dashboard.
- The loop skips checkpoints that already exist on disk, so re-running
  the notebook does not retrain unless you delete the file.

*The cells below only execute when `MODE == "train"`.*


In [ ]:
if not TRAIN_MODE:
    print("Skipping Task 66 training loop (MODE=eval). Switch MODE to \"train\" to run this cell.")
else:
    # Task 66 — enable, then train. EDIT THIS LIST.

    ENABLED_EXPERIMENTS = ["E0"]   # add one experiment per session

    for _name in ENABLED_EXPERIMENTS:
        if _name not in EXPERIMENTS:
            raise KeyError(f"unknown experiment {_name!r}; defined: {list(EXPERIMENTS)}")
        train_experiment(
            name=_name, cfg=EXPERIMENTS[_name],
            windows_aug=windows_aug, ckpt_dir=CKPT_DIR,
            wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
            alpha_np=alpha_np, sigma_np=sigma_np, T=T,
            bin_centers=BIN_CENTERS, bin_width=BIN_WIDTH,
        )


---
## Part D — Visual sanity check on the most recent training

Sample a small batch of validation conditioning vectors and overlay
the diffusion's generated residuals against the ground truth. This is
a "did training collapse?" check — not a quantitative comparison. The
real evaluation lives in the EVAL MODE section below.


In [ ]:
if not TRAIN_MODE:
    print("Skipping Part D visual sanity check (MODE=eval). Switch MODE to \"train\" to run this cell.")
else:
    # Part D — quick overlay for the most recently trained checkpoint.

    if not ENABLED_EXPERIMENTS:
        print("No experiments were trained this session — nothing to visualize.")
    else:
        _name = ENABLED_EXPERIMENTS[-1]
        _cfg  = EXPERIMENTS[_name]
        lit, _, val_ds, _ = load_trained_experiment(
            _name, _cfg, windows_aug, CKPT_DIR, alpha_np, sigma_np,
        )

        n_show = 4
        cond_concat = torch.cat(
            [torch.stack([val_ds[i][k] for i in range(n_show)])
             for k in _cfg["consumed_keys"]],
            dim=-1,
        )
        truth = torch.stack([val_ds[i]["r_clean"] for i in range(n_show)]).numpy()
        truth_phys = truth * val_ds.bin_stds.numpy() + val_ds.bin_means.numpy()
        samples = sample_conditional_extended(lit, cond_concat, guidance_w=0.0).cpu().numpy()

        fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3.2), sharey=True)
        for i, ax in enumerate(axes):
            w = BIN_WIDTH * 0.4
            ax.bar(BIN_CENTERS - w / 2, truth_phys[i], width=w, color="C0", label="truth")
            ax.bar(BIN_CENTERS + w / 2, samples[i],    width=w, color="C2", label="sampled")
            ax.axhline(0, color="k", lw=0.4)
            ax.set_title(f"val window {i}")
            ax.set_xlabel("|latitude| (°)")
        axes[0].set_ylabel("residual"); axes[0].legend()
        fig.suptitle(f"{_name}: visual sanity check")
        fig.tight_layout(); plt.show()


<a id="eval-mode"></a>

---
# Evaluation mode — NLL ablations across all experiments (Tasks 67–70)

This is the evaluation half of the Week 10 extension. It scores every
checkpoint produced by train mode (`ckpt_E*.ckpt` in this directory)
against the same **hard-gated NLL** metric used in 10c, and adds a
critical new diagnostic: an **oracle MLP** that maps each experiment's
cond vector directly to per-bin Gaussian residual parameters. The
oracle's NLL is an *upper bound* on what any model can extract from a
given cond set:

- Diffusion ≪ oracle  →  architecture is the bottleneck (try FiLM, CFG, Fourier).
- Oracle ≈ classical  →  the cond set itself doesn't help; try a different
  cond group or stop running that variant.

This is what makes the ablation scientifically honest: without an
oracle, a flat NLL across experiments could mean *either* "more cond
doesn't help" *or* "the architecture can't extract the new cond's
information" — two completely different fixes.

*The code cells below only execute when `MODE == "eval"`.*


---
## Task 67 — Build per-window evaluation blocks

Re-window the raw CSV the same way 10c does (per-window, 6-monthly,
≥ 20 obs per window) and tag each window with its v2 parquet row's
*entire* cond superset — every group, normalized later per-experiment
using the corresponding checkpoint's `cond_<g>_means` / `cond_<g>_stds`
buffers. We work with **per-window blocks only** in 10e — that is the
granularity at which the diffusion model is native, and the granularity
where any improvement over Week 10 will be most visible.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Task 67 — Build per-window evaluation blocks + NLL primitives + Oracle MLP
# ═══════════════════════════════════════════════════════════════════════════

import torch.optim as optim
from scipy.stats import norm as sp_norm

# ── Part 1: build per-window evaluation blocks ────────────────────────────
print("── Task 67 Part 1: rebuild per-window evaluation blocks ─────────────")

raw_df_eval = pd.read_csv(data_path, parse_dates=[[0,1,2]], keep_date_col=False)
raw_df_eval.rename(columns={"year_month_day":"date"}, inplace=True)
raw_df_eval = raw_df_eval[raw_df_eval["latitude"].notna()].copy()
raw_df_eval["hemisphere"] = raw_df_eval["latitude"].apply(
    lambda v: "north" if v >= 0 else "south")
raw_df_eval["abs_lat"]    = raw_df_eval["latitude"].abs()
raw_df_eval["year"]       = raw_df_eval["date"].dt.year
raw_df_eval = raw_df_eval.dropna(subset=["CYCLE"]).copy()
raw_df_eval["CYCLE"]      = raw_df_eval["CYCLE"].astype(int)

# Pull split tags from v2 parquet (cycle-level)
hc_split_eval = (windows_aug.groupby(["cycle","hemisphere"])["split"]
                             .agg(lambda s: s.iloc[0]).to_dict())
amp_lookup_eval = (windows_aug.groupby(["cycle","hemisphere"])["amplitude"]
                              .agg("first").to_dict())

# Build per-block cond lookup from v2 parquet keyed by (cycle, hemi, tau_center)
# We convert tau_center → approximate calendar year for matching
_t0_by_hc_eval = {}
for (cyc, hemi) in hc_split_eval:
    t0 = t0_refined.get((cyc, hemi), np.nan)
    if not np.isnan(t0):
        _t0_by_hc_eval[(int(cyc), str(hemi))] = float(t0)

# Build lookup: (cycle, hemi, approx_calendar_year) → full cond row
_all_group_cols = (GROUP_COLS["base"] + GROUP_COLS["cyclehemi"] +
                   GROUP_COLS["opp"]  + GROUP_COLS["traj"])
_all_group_cols = [c for c in _all_group_cols if c in windows_aug.columns]

_cond_lookup_eval = {}
for _, row in windows_aug.iterrows():
    cyc  = int(row["cycle"]); hemi = str(row["hemisphere"])
    t0   = _t0_by_hc_eval.get((cyc, hemi), np.nan)
    if np.isnan(t0): continue
    cal_year = float(row["tau_center"]) + t0   # approximate calendar year
    _cond_lookup_eval[(cyc, hemi, round(cal_year, 4))] = {
        col: float(row[col]) for col in _all_group_cols if col in row.index
    }

def _lookup_cond_eval(cyc, hemi, c_dec, tol=0.05):
    """Closest match in calendar-year units. Returns dict or None."""
    best_v, best_d = None, tol
    for (c2, h2, yc), v in _cond_lookup_eval.items():
        if c2 != cyc or h2 != hemi: continue
        d = abs(yc - c_dec)
        if d <= best_d:
            best_v, best_d = v, d
    return best_v

def _build_per_window_hc_eval(cyc, hemi, df_hc, t0):
    if len(df_hc) == 0: return []
    y0 = df_hc["date"].min().year; y1 = df_hc["date"].max().year + 1
    bounds = sorted({pd.Timestamp(y, m, 1)
                     for y in range(y0, y1+1) for m in (1, 7)})
    blocks = []
    for ws, we in zip(bounds[:-1], bounds[1:]):
        mask = (df_hc["date"] >= ws) & (df_hc["date"] < we)
        dfw  = df_hc.loc[mask]
        if len(dfw) < 20: continue
        c_ts  = ws + (we - ws) / 2
        c_dec = c_ts.year + c_ts.timetuple().tm_yday / 365.25
        tau   = c_dec - t0
        cond_raw = _lookup_cond_eval(cyc, hemi, c_dec)
        if cond_raw is None: continue
        # Build per-group raw cond dict
        groups_raw = {}
        for grp, cols in GROUP_COLS.items():
            vals = [cond_raw.get(c, np.nan) for c in cols if c in _all_group_cols]
            if all(np.isfinite(v) for v in vals) and len(vals) == len(
                    [c for c in cols if c in _all_group_cols]):
                groups_raw[grp] = np.array(vals, dtype=np.float32)
        if "base" not in groups_raw: continue
        blocks.append({
            "center_decimal": float(c_dec),
            "tau"           : float(tau),
            "lats"          : dfw["abs_lat"].to_numpy(np.float32),
            "groups_raw"    : groups_raw,
        })
    return blocks

hemicycles = []
for (cyc, hemi), split in hc_split_eval.items():
    if split not in ("train","val"): continue
    t0 = _t0_by_hc_eval.get((int(cyc), str(hemi)), np.nan)
    if np.isnan(t0): continue
    A   = float(amp_lookup_eval.get((cyc, hemi), np.nan))
    if np.isnan(A): continue
    df_hc = raw_df_eval[(raw_df_eval["CYCLE"]==int(cyc)) &
                         (raw_df_eval["hemisphere"]==hemi)]
    blocks = _build_per_window_hc_eval(cyc, hemi, df_hc, t0)
    if not blocks: continue
    hemicycles.append({
        "cycle"     : int(cyc), "hemisphere": hemi,
        "amplitude" : A,        "t0"        : t0,
        "split"     : split,    "blocks"    : blocks,
    })

built_n = sum(len(hc["blocks"]) for hc in hemicycles)
parq_n  = ((windows_aug["split"]=="train")|(windows_aug["split"]=="val")).sum()
print(f"per-window blocks: built={built_n}  parquet train+val={parq_n}  "
      f"({'match' if built_n==parq_n else 'mismatch'})")
print(f"hemicycles: {len(hemicycles)}")

assert len(hemicycles) > 0, "No blocks built"
assert all("groups_raw" in blk
           for hc in hemicycles for blk in hc["blocks"]), \
    "Every block needs groups_raw"

---
## Task 67 (cont) — NLL primitives, same as 10c

These are byte-identical to the primitives in 10c. Re-stated here so
10e can be run standalone (without executing 10c first).


In [ ]:
# ── Part 2: NLL primitives (port from 10c) ────────────────────────────────
print("\n── Task 67 Part 2: NLL primitives ───────────────────────────────────")

def hard_nll_classical(model, hcs):
    total, included, candidate = 0.0, 0, 0
    for hc in hcs:
        A    = hc["amplitude"]
        mu0A = float(model.mu_0(A)) if hasattr(model, "mu_0") else 45.0
        for blk in hc["blocks"]:
            candidate += 1
            mu = float(exp_decay(blk["tau"], a_mu_univ, b_mu_univ))
            if mu > mu0A: continue
            sigma_v = m_shared_fit * mu + b_shared_fit
            if sigma_v <= 0: continue
            ll = sp_norm.logpdf(blk["lats"], loc=mu, scale=sigma_v).mean()
            if not np.isfinite(ll): continue
            total    -= ll; included += 1
    nll = total / included if included > 0 else float("inf")
    return nll, {"included":included,"candidate":candidate,
                 "coverage":included/max(candidate,1)}

def hard_nll_combined(model, hcs, residuals_by_block, eps=1e-6):
    total, included, candidate = 0.0, 0, 0
    n_floored, n_lats = 0, 0
    for hc in hcs:
        A    = hc["amplitude"]
        mu0A = float(model.mu_0(A)) if hasattr(model, "mu_0") else 45.0
        for blk in hc["blocks"]:
            candidate += 1
            mu = float(exp_decay(blk["tau"], a_mu_univ, b_mu_univ))
            if mu > mu0A: continue
            sigma_v = m_shared_fit * mu + b_shared_fit
            if sigma_v <= 0: continue
            key = (hc["cycle"], hc["hemisphere"], blk["center_decimal"])
            if key not in residuals_by_block: continue
            residual = residuals_by_block[key]
            lats     = blk["lats"]
            p_cl     = sp_norm.pdf(lats, loc=mu, scale=sigma_v)
            bin_ix   = np.clip(np.floor(lats/BIN_WIDTH).astype(int), 0, 14)
            p_raw    = p_cl + residual[bin_ix]
            p_comb   = np.maximum(eps, p_raw)
            n_floored += int((p_raw < eps).sum()); n_lats += len(lats)
            ll = np.log(p_comb).mean()
            if not np.isfinite(ll): continue
            total    -= ll; included += 1
    nll = total / included if included > 0 else float("inf")
    return nll, {"included":included,"candidate":candidate,
                 "coverage":included/max(candidate,1),
                 "floor_fraction":n_floored/max(n_lats,1)}

# Smoke test
val_hcs = [hc for hc in hemicycles if hc["split"]=="val"]
nll_cl_val, det_cl = hard_nll_classical(None, val_hcs)
print(f"classical hard NLL (val): {nll_cl_val:.4f}  "
      f"coverage={det_cl['coverage']:.3f}")

---
## Task 67 (cont) — Diagnostic oracle

For each experiment's cond set, fit a tiny MLP that maps the cond
vector directly to a 15-D Gaussian over the residual bins
(`mean`, `log_std`). The oracle's NLL is computed by sampling K
residuals from the per-block Gaussian and feeding them through
`hard_nll_combined` — the same harness used to score the diffusion.

What the oracle answers:

- **Diffusion ≪ oracle**  →  the diffusion isn't extracting the
  information that's already in the cond. Try a stronger architecture
  (FiLM, Fourier features, larger MLP).
- **Oracle ≈ classical**  →  the cond set itself doesn't carry enough
  information about the residual structure. Try a different cond
  group or stop adding to this one.

You implement this. The science (the tiny MLP, the Gaussian NLL
expression, the training loop, sampling from the predicted Gaussian)
is yours.

In [ ]:
# ── Part 3: Oracle MLP ────────────────────────────────────────────────────
print("\n── Task 67 Part 3: Oracle MLP ───────────────────────────────────────")

class OracleMLP(nn.Module):
    """
    Maps a cond vector → (mean, log_std) for 15 residual bins.
    Fits a conditional Gaussian distribution p(r | cond) as an upper
    bound on what any model can extract from a given cond set.
    """
    def __init__(self, cond_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 30),   # 15 means + 15 log_stds
        )
    def forward(self, cond):
        out      = self.net(cond)               # (B, 30)
        mean     = out[:, :15]                  # (B, 15)
        log_std  = out[:, 15:].clamp(-4, 4)    # (B, 15)
        return mean, log_std

def gaussian_nll(r, mean, log_std):
    """
    Per-row Gaussian NLL of standardized residuals.
    r, mean, log_std: (B, 15) tensors.
    Returns scalar mean NLL.
    """
    # -log N(r; mean, exp(log_std))
    # = log_std + 0.5*(r-mean)^2 / exp(2*log_std) + 0.5*log(2π)
    nll = log_std + 0.5 * ((r - mean) / log_std.exp()) ** 2
    return nll.mean()

def fit_oracle(cond_train, r_train, cond_val, r_val,
               max_epochs=2000, lr=1e-2, hidden_dim=64, seed=0):
    """
    Fit OracleMLP on standardized (cond, residual) pairs.

    Parameters
    ----------
    cond_train, cond_val : np.ndarray (N, cond_dim)
    r_train, r_val       : np.ndarray (N, 15) — standardized residuals
    Returns: (oracle_module, best_val_nll)
    """
    torch.manual_seed(seed)
    cond_dim = cond_train.shape[1]
    oracle   = OracleMLP(cond_dim=cond_dim, hidden_dim=hidden_dim)
    opt      = optim.Adam(oracle.parameters(), lr=lr)

    # Convert to tensors
    ct = torch.tensor(cond_train, dtype=torch.float32)
    rt = torch.tensor(r_train,    dtype=torch.float32)
    cv = torch.tensor(cond_val,   dtype=torch.float32)
    rv = torch.tensor(r_val,      dtype=torch.float32)

    best_val_nll = float("inf")
    best_state   = None
    patience     = 200
    no_improve   = 0

    oracle.train()
    for epoch in range(max_epochs):
        opt.zero_grad()
        mean, log_std = oracle(ct)
        loss          = gaussian_nll(rt, mean, log_std)
        loss.backward()
        opt.step()

        if epoch % 20 == 0:
            oracle.eval()
            with torch.no_grad():
                vm, vl = oracle(cv)
                val_nll = gaussian_nll(rv, vm, vl).item()
            oracle.train()
            if val_nll < best_val_nll - 1e-6:
                best_val_nll = val_nll
                best_state   = {k: v.clone() for k, v in oracle.state_dict().items()}
                no_improve   = 0
            else:
                no_improve  += 1
            if no_improve >= patience // 20:
                break

    if best_state is not None:
        oracle.load_state_dict(best_state)
    oracle.eval()
    return oracle, best_val_nll

print("✓ Task 67 complete")
print(f"  hard_nll_classical, hard_nll_combined defined")
print(f"  OracleMLP, gaussian_nll, fit_oracle defined")
print(f"  hemicycles: {len(hemicycles)} ({built_n} blocks)")

---
## Task 68 — Score every checkpoint

For each discovered `ckpt_E*.ckpt`:

1. Build per-experiment cond tensors for every val block by
   concatenating the right groups in `consumed_keys` order, normalized
   with the **checkpoint's own** per-group buffers (so val data uses
   train-set normalization recovered from the saved model).
2. Run K = 100 conditional samples per block using
   `sample_conditional_extended`. For E6 (CFG), repeat the sampling at
   every guidance weight in `CFG_GUIDANCE_W` and keep them as separate
   rows.
3. Fit the oracle MLP on the same (cond, standardized residual) data
   and record its val NLL as the upper bound for this cond set.
4. Plug each of the K samples into `hard_nll_combined`; report mean
   and σ over K.


In [ ]:
# ── Guard: ensure EXPERIMENTS and CKPT_DIR are defined ───────────────────
if "EXPERIMENTS" not in dir():
    print("EXPERIMENTS not defined — re-running setup definitions...")
    _BASE_TEMPLATE = {
        "arch"          : "concat",
        "consumed_keys" : ["cond_base"],
        "groups"        : ["base"],
        "hidden_dim"    : 128,
        "n_layers"      : 3,
        "fourier"       : False,
        "cond_dropout_p": 0.0,
        "max_epochs"    : 20000,
        "lr"            : 1e-3,
        "batch_size"    : 64,
    }
    def _spec(**overrides):
        d = dict(_BASE_TEMPLATE); d.update(overrides); return d
    EXPERIMENTS = {
        "E0": _spec(),
        "E1": _spec(consumed_keys=["cond_base","cond_cyclehemi"],
                    groups=["base","cyclehemi"]),
        "E2": _spec(consumed_keys=["cond_base","cond_opp"],
                    groups=["base","opp"]),
        "E3": _spec(consumed_keys=["cond_base","cond_traj"],
                    groups=["base","traj"]),
        "E4": _spec(arch="film"),
        "E5": _spec(fourier=True),
        "E6": _spec(cond_dropout_p=0.1),
    }

if "CKPT_DIR" not in dir():
    CKPT_DIR = "."

if "K_EVAL" not in dir():
    K_EVAL = 100

if "CFG_GUIDANCE_W" not in dir():
    CFG_GUIDANCE_W = [0.0, 1.0, 2.0, 3.0]

print(f"EXPERIMENTS: {list(EXPERIMENTS.keys())}")
print(f"CKPT_DIR   : {CKPT_DIR}")
print(f"K_EVAL     : {K_EVAL}")

# ═══════════════════════════════════════════════════════════════════════════
# Task 68 — Score every checkpoint
# ═══════════════════════════════════════════════════════════════════════════

K_EVAL         = 100
CFG_GUIDANCE_W = [0.0, 1.0, 2.0, 3.0]
CACHE_PATH_68  = "./task68_samples_cache.npz"

def _load_experiment_checkpoint(name, cfg):
    """Load a trained ExtendedConditionalDiffusionLightning checkpoint."""
    ckpt_path = os.path.join(CKPT_DIR, f"ckpt_{name}.ckpt")
    if not os.path.exists(ckpt_path):
        print(f"  {name}: checkpoint not found at {ckpt_path}")
        return None, None, None

    cond_dim = _compute_cond_dim(cfg, ext_train_ds)
    model = ExtendedConditionalDiffusionMLP(
        data_dim    = 15,
        hidden_dim  = cfg["hidden_dim"],
        t_embed_dim = 64,
        t_hidden_dim= 128,
        cond_dim    = cond_dim,
        n_layers    = cfg["n_layers"],
        arch        = cfg["arch"],
        fourier     = cfg["fourier"],
    )
    # Build placeholder dicts matching the checkpoint's buffer shapes
    cm_by_g = {g: ext_train_ds.cond_means_by_group[g].numpy()
               for g in ext_train_ds.cond_means_by_group}
    cs_by_g = {g: ext_train_ds.cond_stds_by_group[g].numpy()
               for g in ext_train_ds.cond_stds_by_group}

    lit = ExtendedConditionalDiffusionLightning.load_from_checkpoint(
        ckpt_path,
        model                = model,
        alpha                = alpha_np,
        sigma                = sigma_np,
        T                    = T,
        lr                   = cfg["lr"],
        consumed_keys        = cfg["consumed_keys"],
        bin_means            = ext_train_ds.bin_means,
        bin_stds             = ext_train_ds.bin_stds,
        cond_means_by_group  = cm_by_g,
        cond_stds_by_group   = cs_by_g,
        map_location         = DEVICE,
    )
    lit.eval().to(DEVICE)
    return lit, ext_train_ds, ext_val_ds


def _block_cond_concat_for_eval(hcs, lit, cfg):
    """
    For each block in hcs, build the normalized, concatenated cond
    tensor matching the experiment's consumed_keys.
    Returns (keys, cond_tensor (N, total_cond_dim)).
    """
    keys, rows = [], []
    for hc in hcs:
        for blk in hc["blocks"]:
            keys.append((hc["cycle"], hc["hemisphere"], blk["center_decimal"]))
            parts = []
            for key in cfg["consumed_keys"]:
                grp   = key.replace("cond_","")
                raw   = blk["groups_raw"].get(grp)
                if raw is None:
                    raw = np.zeros(len(GROUP_COLS.get(grp,[])), dtype=np.float32)
                cm  = getattr(lit, f"cond_means_{grp}", torch.zeros(len(raw)))
                cs  = getattr(lit, f"cond_stds_{grp}",  torch.ones(len(raw)))
                parts.append((raw - cm.cpu().numpy()) / cs.cpu().numpy())
            rows.append(np.concatenate(parts))
    cond_tensor = torch.tensor(np.stack(rows), dtype=torch.float32)
    return keys, cond_tensor


def _k_run_combined_from_array(hcs, sample_keys, sample_arr_NK15):
    """K-run hard_nll_combined from pre-sampled array."""
    N, K_loc, _ = sample_arr_NK15.shape
    nlls   = np.empty(K_loc)
    floors = np.empty(K_loc)
    for k in range(K_loc):
        res_dict = {sample_keys[i]: sample_arr_NK15[i, k]
                    for i in range(len(sample_keys))}
        nll, det = hard_nll_combined(None, hcs, res_dict)
        nlls[k]   = nll
        floors[k] = det["floor_fraction"]
    return nlls, floors


def _get_oracle_data(hcs, cfg, lit):
    """
    Build (cond_raw_norm, r_std) arrays for oracle fitting.
    Uses the *concatenated* cond matching consumed_keys.
    """
    cond_rows, r_rows = [], []
    emp_cols_l = [f"hist_emp_{k:02d}" for k in range(15)]
    par_cols_l = [f"hist_par_{k:02d}" for k in range(15)]

    bm = lit.bin_means.cpu().numpy()
    bs = lit.bin_stds.cpu().numpy()

    for hc in hcs:
        for blk in hc["blocks"]:
            parts = []
            for key in cfg["consumed_keys"]:
                grp = key.replace("cond_","")
                raw = blk["groups_raw"].get(grp)
                if raw is None:
                    raw = np.zeros(len(GROUP_COLS.get(grp,[])), dtype=np.float32)
                cm = getattr(lit, f"cond_means_{grp}", torch.zeros(len(raw)))
                cs = getattr(lit, f"cond_stds_{grp}",  torch.ones(len(raw)))
                parts.append((raw - cm.cpu().numpy()) / cs.cpu().numpy())
            cond_rows.append(np.concatenate(parts))
            # Residual = emp hist - par hist for this block
            # We need it from the parquet row matching this block
            key = (hc["cycle"], hc["hemisphere"], blk["center_decimal"])
            # Look up from windows_aug
            match = windows_aug[
                (windows_aug["cycle"]==hc["cycle"]) &
                (windows_aug["hemisphere"]==hc["hemisphere"])
            ]
            if len(match) == 0:
                r_rows.append(np.zeros(15, dtype=np.float32))
                continue
            # Find closest tau_center
            t0  = hc["t0"]
            tau = blk["tau"]
            match = match.copy()
            match["_diff"] = (match["tau_center"] - tau).abs()
            best = match.nsmallest(1,"_diff").iloc[0]
            emp  = best[[f"hist_emp_{k:02d}" for k in range(15)]].values.astype(np.float32)
            par  = best[[f"hist_par_{k:02d}" for k in range(15)]].values.astype(np.float32)
            r_phys = emp - par
            r_std  = (r_phys - bm) / bs
            r_rows.append(r_std)

    return np.stack(cond_rows), np.stack(r_rows)


def score_checkpoint(name, cfg):
    """Score one experiment. Returns list of result dicts."""
    print(f"\n  Scoring {name}...")
    lit, tr_ds, vl_ds = _load_experiment_checkpoint(name, cfg)
    if lit is None:
        return []

    val_hcs_score = [hc for hc in hemicycles if hc["split"]=="val"]
    keys_v, cond_v = _block_cond_concat_for_eval(val_hcs_score, lit, cfg)

    # ── Step 1: determine guidance values to sweep ────────────────────────
    guidance_vals = (CFG_GUIDANCE_W if cfg["cond_dropout_p"] > 0
                     else [0.0])

    rows_out = []
    for guidance_w in guidance_vals:
        print(f"    guidance_w={guidance_w:.1f} ...")

        # ── Step 2: generate K conditional samples ────────────────────────
        cond_K = repeat(cond_v, "n d -> (n k) d", k=K_EVAL)
        torch.manual_seed(0)
        samples_raw = sample_conditional_extended(
            lit, cond_K, guidance_w=guidance_w,
            data_dim=15, device=DEVICE)   # (N*K, 15) physical units
        samples = samples_raw.reshape(len(keys_v), K_EVAL, 15)

        # ── Step 3: compute K NLLs via hard_nll_combined ──────────────────
        nlls, floors = _k_run_combined_from_array(val_hcs_score, keys_v, samples)

        # ── Step 4: oracle upper bound ────────────────────────────────────
        train_hcs_score = [hc for hc in hemicycles if hc["split"]=="train"]
        cond_tr, r_tr   = _get_oracle_data(train_hcs_score, cfg, lit)
        cond_vl, r_vl   = _get_oracle_data(val_hcs_score,   cfg, lit)

        oracle, oracle_val_nll_std = fit_oracle(
            cond_tr, r_tr, cond_vl, r_vl,
            max_epochs=2000, lr=1e-2, hidden_dim=64)

        # Sample K residuals from oracle Gaussian → hard_nll_combined
        oracle_cond_t = torch.tensor(cond_vl, dtype=torch.float32)
        with torch.no_grad():
            o_mean, o_log_std = oracle(oracle_cond_t)
        o_mean_np    = o_mean.numpy()    # (N_val, 15) standardized
        o_log_std_np = o_log_std.numpy()

        # De-standardize oracle samples to physical units
        bm = lit.bin_means.cpu().numpy()
        bs = lit.bin_stds.cpu().numpy()
        rng_oracle = np.random.default_rng(42)
        oracle_samples = (o_mean_np[None] +
                          np.exp(o_log_std_np[None]) *
                          rng_oracle.standard_normal((K_EVAL,)+o_mean_np.shape)
                         )  # (K, N_val, 15) standardized
        oracle_samples = oracle_samples * bs + bm  # physical units
        oracle_samples = oracle_samples.transpose(1, 0, 2)  # (N_val, K, 15)

        oracle_nlls, oracle_floors = _k_run_combined_from_array(
            val_hcs_score, keys_v, oracle_samples)

        rows_out.append({
            "experiment"     : name,
            "guidance_w"     : guidance_w,
            "nll_mean"       : float(nlls.mean()),
            "nll_std"        : float(nlls.std()),
            "floor"          : float(floors.mean()),
            "oracle_nll_mean": float(oracle_nlls.mean()),
            "oracle_nll_std" : float(oracle_nlls.std()),
            "oracle_gauss"   : float(oracle_val_nll_std),
            "coverage"       : float(len(keys_v)) / max(
                sum(len(hc["blocks"]) for hc in val_hcs_score), 1),
        })

        print(f"      NLL={nlls.mean():.4f}±{nlls.std():.4f}  "
              f"oracle={oracle_nlls.mean():.4f}  "
              f"floor={floors.mean():.4f}")

    return rows_out


# ── Run scoring ───────────────────────────────────────────────────────────
available_ckpts = [
    name for name in EXPERIMENTS
    if os.path.exists(os.path.join(CKPT_DIR, f"ckpt_{name}.ckpt"))
]
print(f"Available checkpoints: {available_ckpts}")

if not available_ckpts:
    print("No checkpoints found. Run Task 66 (train mode) first.")
    scoreboard = pd.DataFrame()
else:
    all_rows = []
    for name in available_ckpts:
        all_rows.extend(score_checkpoint(name, EXPERIMENTS[name]))

    scoreboard = pd.DataFrame(all_rows)
    nll_cl_val_67, _ = hard_nll_classical(None, val_hcs)
    scoreboard["classical"] = nll_cl_val_67
    scoreboard["delta"]     = scoreboard["nll_mean"] - nll_cl_val_67

    print("\n" + "="*90)
    print("TASK 68 SCOREBOARD")
    print("="*90)
    print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("="*90)

print("\n✓ Task 68 complete")

In [ ]:
if not EVAL_MODE:
    print("Skipping Task 68 scoring loop (MODE=train). Switch MODE to \"eval\" once you have ckpt_E*.ckpt files.")
else:
    # Discover trained checkpoints and score them.
    _ckpts = discover_experiment_checkpoints(_WEEK10_DIR)
    for name in _ckpts:
        assert name in EXPERIMENTS, (
            f"ckpt_{name}.ckpt has no entry in EXPERIMENTS — add a spec to Task 65 "
            f"(both halves of this notebook share the same dict)."
        )
    print(f"discovered checkpoints: {list(_ckpts)}")

    all_rows = []
    for name in _ckpts:
        print(f"scoring {name} ...")
        all_rows.extend(score_checkpoint(name, EXPERIMENTS[name]))

    scoreboard = pd.DataFrame(all_rows)
    scoreboard["classical"] = nll_cl_val
    print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


---
## Task 69 — Headline plot

Bar chart, val split: classical baseline plus every experiment's NLL,
with K-σ error bars. Oracle NLL per experiment overlaid as a
horizontal dashed marker to make the "what's achievable from this cond
set" boundary visible.

For any CFG variant (`cond_dropout_p > 0`), the bar shown is the
best-NLL guidance setting; a secondary panel sweeps the guidance
weight `w` so you can see the guidance vs NLL trade.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Task 69 — Headline plot + CFG sweep + per-hemicycle breakdown
# ═══════════════════════════════════════════════════════════════════════════

if scoreboard is None or len(scoreboard) == 0:
    print("No scoreboard data — run Task 68 first.")
else:
    nll_cl = scoreboard["classical"].iloc[0]
    val_hcs_69 = [hc for hc in hemicycles if hc["split"] == "val"]

    # ── Panel 1: headline bar chart ───────────────────────────────────────
    # For CFG experiments, show only the best guidance weight
    best_rows = []
    for name in scoreboard["experiment"].unique():
        sub = scoreboard[scoreboard["experiment"] == name]
        best = sub.nsmallest(1, "nll_mean").iloc[0]
        best_rows.append(best)
    best_df = pd.DataFrame(best_rows).reset_index(drop=True)

    n_exp   = len(best_df)
    x       = np.arange(n_exp + 1)   # +1 for classical
    w       = 0.55
    labels  = ["classical"] + list(best_df["experiment"])
    nll_vals= [nll_cl] + list(best_df["nll_mean"])
    nll_err = [0.0]    + list(best_df["nll_std"])
    o_vals  = [np.nan] + list(best_df["oracle_nll_mean"])

    fig1, ax1 = plt.subplots(figsize=(max(8, 1.5*len(labels)), 5))
    colors = ["tab:blue"] + ["tab:green" if v < nll_cl else "tab:red"
                              for v in best_df["nll_mean"]]
    bars = ax1.bar(x, nll_vals, yerr=nll_err, width=w,
                   color=colors, alpha=0.75,
                   capsize=4, ecolor="black", linewidth=0.5)

    # Oracle markers as dashed horizontal lines above each experiment bar
    for i, (ov, xi) in enumerate(zip(o_vals[1:], x[1:]), start=1):
        if np.isfinite(ov):
            ax1.hlines(ov, xi - w/2, xi + w/2,
                       colors="darkorange", linewidths=2.0,
                       linestyles="--", label="oracle" if i == 1 else "")

    # Classical reference line
    ax1.axhline(nll_cl, color="tab:blue", linewidth=1.2,
                linestyle=":", alpha=0.7, label="classical")

    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=20, fontsize=9)
    ax1.set_ylabel("hard NLL (val)  — lower is better")
    ax1.set_title(
        "Task 69 — Headline NLL comparison: all experiments vs classical\n"
        "Green = improvement  |  Red = regression  |  "
        "Orange dashed = oracle upper bound for that cond set",
        fontsize=9)
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_els = [
        Patch(color="tab:blue",  alpha=0.75, label="Classical"),
        Patch(color="tab:green", alpha=0.75, label="Diffusion (better)"),
        Patch(color="tab:red",   alpha=0.75, label="Diffusion (worse)"),
        Line2D([0],[0], color="darkorange", linewidth=2, linestyle="--",
               label="Oracle NLL"),
    ]
    ax1.legend(handles=legend_els, fontsize=8, loc="upper right")
    ax1.set_ylim(min(nll_vals + [v for v in o_vals if np.isfinite(v)]) * 0.95,
                 max(nll_vals + [v for v in o_vals if np.isfinite(v)]) * 1.05)
    plt.tight_layout()
    plt.show()

    # ── Panel 2: CFG sweep (only if any CFG experiment exists) ────────────
    cfg_exps = [name for name in scoreboard["experiment"].unique()
                if EXPERIMENTS[name]["cond_dropout_p"] > 0]

    if cfg_exps:
        fig2, ax2 = plt.subplots(figsize=(8, 4))
        for name in cfg_exps:
            sub = scoreboard[scoreboard["experiment"] == name].sort_values("guidance_w")
            ax2.errorbar(sub["guidance_w"], sub["nll_mean"], yerr=sub["nll_std"],
                         marker="o", linewidth=2, capsize=4, label=name)
        ax2.axhline(nll_cl, color="gray", linewidth=1.2, linestyle="--",
                    label="classical")
        ax2.set_xlabel("Guidance weight w")
        ax2.set_ylabel("hard NLL (val)")
        ax2.set_title("CFG guidance weight sweep\n"
                      "Optimal w trades off sharpness vs coverage",
                      fontsize=9)
        ax2.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

    # ── Panel 3: per-hemicycle breakdown for the best experiment ─────────
    best_exp_name = best_df.nsmallest(1, "nll_mean").iloc[0]["experiment"]
    best_cfg      = EXPERIMENTS[best_exp_name]
    best_g_w      = float(best_df.loc[
        best_df["experiment"]==best_exp_name, "guidance_w"].iloc[0])

    print(f"\nPer-hemicycle breakdown for {best_exp_name} "
          f"(guidance_w={best_g_w:.1f})...")

    lit_best, _, _ = _load_experiment_checkpoint(best_exp_name, best_cfg)
    if lit_best is None:
        print("Could not load best checkpoint for breakdown.")
    else:
        cycle_rows_69 = []
        for hc in val_hcs_69:
            # Classical NLL for this single hemicycle
            nll_cl_hc, _ = hard_nll_classical(None, [hc])

            # Sample for this hemicycle
            keys_hc, cond_hc = _block_cond_concat_for_eval([hc], lit_best, best_cfg)
            if len(keys_hc) == 0:
                continue
            cond_hc_K = repeat(cond_hc, "n d -> (n k) d", k=K_EVAL)
            torch.manual_seed(0)
            s_raw = sample_conditional_extended(
                lit_best, cond_hc_K, guidance_w=best_g_w,
                data_dim=15, device=DEVICE)
            s_arr = s_raw.reshape(len(keys_hc), K_EVAL, 15)
            nlls_hc, _ = _k_run_combined_from_array([hc], keys_hc, s_arr)
            cycle_rows_69.append({
                "label"      : f"{hc['cycle']:02d}{hc['hemisphere'][0].upper()}",
                "classical"  : nll_cl_hc,
                "cond_mean"  : nlls_hc.mean(),
                "cond_std"   : nlls_hc.std(),
                "delta"      : nlls_hc.mean() - nll_cl_hc,
            })

        breakdown_df = (pd.DataFrame(cycle_rows_69)
                          .sort_values("delta")
                          .reset_index(drop=True))

        fig3, ax3 = plt.subplots(
            figsize=(max(8, 0.8*len(breakdown_df)), 5))
        x3 = np.arange(len(breakdown_df)); w3 = 0.38
        ax3.bar(x3 - w3/2, breakdown_df["classical"],
                width=w3, color="tab:blue", alpha=0.75, label="classical")
        ax3.bar(x3 + w3/2, breakdown_df["cond_mean"],
                yerr=breakdown_df["cond_std"],
                width=w3, color="tab:green", alpha=0.75,
                capsize=3, ecolor="black", label=best_exp_name)
        ax3.axhline(nll_cl, color="gray", linewidth=1, linestyle="--")
        ax3.set_xticks(x3)
        ax3.set_xticklabels(breakdown_df["label"], rotation=45, fontsize=8)
        ax3.set_ylabel("hard NLL (val)")
        ax3.set_title(
            f"Per-hemicycle NLL breakdown — {best_exp_name} vs classical\n"
            "Sorted by Δ NLL (most improved → least improved)",
            fontsize=9)
        ax3.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        print("\nTop 5 most improved hemicycles:")
        print(breakdown_df[["label","classical","cond_mean","delta"]]
              .head(5).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
        print("\nBottom 5 (least improved / regressed):")
        print(breakdown_df[["label","classical","cond_mean","delta"]]
              .tail(5).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\n✓ Task 69 complete")

---
## Task 70 — Going further

Once you've worked through the Level 1–5 escalation menu in 10d and
want to push further, the tiered menu below ranks the next experiments
by expected payoff per unit effort. Discipline still applies: one knob
at a time, log to wandb, add to `EXPERIMENTS` in both 10d and 10e,
then re-run.

**Level 1 — easy wins**
- **Wider/deeper MLP.** Bump `hidden_dim` from 128 to 256, `n_layers`
  from 3 to 5 in the winning experiment's config. If NLL drops, the
  network was capacity-bound — interesting on its own.
- **Longer K at evaluation.** Bump K from 100 to 500 for the winning
  variant — tightens the K-σ error bar and lets you trust smaller
  margins.
- **Sampler comparison.** Re-score the winner with DDPM (stochastic)
  sampling instead of the deterministic DDIM in
  `sample_conditional_extended`. Deterministic samplers can under-
  disperse, inflating NLL.

**Level 2 — extra cond information**
- **Larger trajectory K.** Bump `K_LAGS` from 4 to 8 in 10d. If the
  trajectory variant's oracle improves but its diffusion doesn't, the
  architecture is underusing the longer history.
- **Lagged opposite-hemisphere.** Pair the trajectory cond with the
  opposite hemisphere — `opp_area_smoothed_lag1..lag4`.

**Level 3 — architectural changes**
- **Cross-attention conditioning.** Replace the FiLM mechanism with
  cross-attention over a small set of learned cond tokens — overkill
  for the cond dim here, but worth knowing if the FiLM gain saturates.
- **Per-bin-aware loss.** Weight the ε-prediction loss by the inverse
  per-bin std so well-resolved bins don't dominate gradients.

**The test set is the PI's.** Every iteration above is val-only. The
final test-set reveal happens once, after the program is closed.

---
## Handoff back to the PI

The headline numbers in `scoreboard` answer two questions:

1. **Does any variant beat the classical baseline on val?** If yes, the
   diffusion approach has earned its place in the final pipeline.
2. **Where is the bottleneck — information or architecture?** Compare
   each row's `nll_mean` to its `oracle_nll_mean`. A large gap means
   the cond set has more information than the diffusion is extracting
   (architecture-bound). A small gap with the oracle near classical
   means the cond set isn't carrying enough information — that line of
   experiments is exhausted; try a different cond group.

The PI will run the test set evaluation on whatever variant the val
results recommend.
